# Week 8: Capstone Project: Protein Identification

Congratulations on reaching the final module of the course! In this final gauntlet, you will apply the skills you've acquired throughout HMB201, HMB301, and HMB491 to identify characteristics of proteins from their amino acid sequences.

## Overview
- Introduction to Foundation Models
- Introduction to Tokenization
- Introduction to Multi-Class Classification
- Picking your Problem
- Present your Results

## Learning Objectives

By the end of this module, you will be able to:
1. Understand the basics of foundation models and their applications in protein identification;
2. Learn about tokenization in the context of protein sequences;
3. Retrieve embeddings from a foundation model for protein sequences;
4. Apply multi-class classification techniques to identify protein characteristics from foundation model embeddings;
5. Use a variety of statistics to evaluate your model's classification performance.

## Introduction to Foundation Models

**Foundation Models** are large-scale deep learning models that have been *pre-trained* on vast amounts of data. They serve as a backbone for a variety of downstream tasks, allowing users to fine-tune them for specific applications with relatively less data and computational resources.

Below is an example of a generic *multimodal* foundation model.
<figure>
<img src="images/fm.png" alt="Generic Foundation Model" width="600">
<figcaption><strong>Figure 1:</strong> A generic multimodal foundation model that can process and generate text, images, and other data types.</figcaption>
</figure>

Tools like ChatGPT are considered foundation models because they have been trained on a wide variety of text data and can be adapted for numerous language tasks.

Generally, the output of a foundation model will be a vector or matrix encoding of the input data that can then be fed into a separate *'head'* for specific tasks like classification or regression.

Next, let's explore some foundation models in the field of life sciences.

### Foundation Models in AI4Science

**AI4Science** is a field that looks to use *artificial intelligence* to solve large, unanswered problems in the natural sciences. Many large biotech companies and research institutions have begun to build foundation models to begin solving problems in biology, chemistry, and medicine.

A landmark moment in AI4Science was the release of *AlphaFold* by DeepMind in 2018. AlphaFold was able to predict the 3D structure of proteins from their amino acid sequences with very high accuracy.
<figure>
<img src="images/alphafold.png" alt="Overview of AlphaFold1" width="1000">
<figcaption><strong>Figure 2:</strong> An overview of AlphaFold1.</figcaption>
</figure>

While the original AlphaFold isn't generally considered a foundation model (since it was specifically only trained for predicting protein structures), AlphaFold3 is considered a foundation model since it has been trained on a variety of protein-related tasks.

A general characteristic of foundation models is their ability to encode information about samples they are trained on. For example, if a downstream task is to predict the structure of a protein, the foundation model's weights will have learned to encode information about protein structures in general.

### What a Foundation Model for Pathology Might Look Like

To better motivate what a foundation model looks like in a biological context, we present a hypothetical example of a unimodal foundation model for pathology.
<figure>
<img src="images/pathology_fm.png" alt="Generic Pathology Foundation Model" width="600">
<figcaption><strong>Figure 3:</strong> A hypothetical unimodal foundation model for pathology.</figcaption>
</figure>

In this example, our inputs are pathology slides and tiles. (Remember we dealt with tiles in Modules 6 & 7!) The foundation model itself is often a large transformer-based architecture. These foundation models are trained on millions of slides, many of which are annotated with various labels like cancer subtype, patient survival post-resection, biomarkers used for identification, and more. The outputs of the foundation model are called **embeddings**. They are then fed into separate networks for specific tasks, such as cancer subtype classification and the regression of patient survival time. This results in multiple **loss** signals, all of which can be used to train the foundation model. The central idea behind this is that the output of the foundation model will contain **good and diverse** representations of the input data, which can then be used for a variety of downstream tasks.

Later, when we use new data **that is not in the foundation model's training set**, we can get embeddings for our new data.

### EvolutionaryScale Model (ESM)

**ESM3**, a Protein Language Model, is a multimodal foundation model developed by EvolutionaryScale. They are a spin-off company from FAIR, Meta's AI research lab. ESM3 is a model that is able to reason across the *sequence*, *structure*, and *function* of proteins.
<figure>
<img src="images/esm3.jpg" alt="Overview of ESM3" width="600">
<figcaption><strong>Figure 4:</strong> An overview of ESM3.</figcaption>
</figure>

For us, this means that the model has learned to encode information about protein sequences, 3D structures, and biological functions. In this module, we will be using ESM2, an older but open-source version of ESM3, to extract embeddings from protein sequences for your choice of a downstream task.

## Introduction to Tokenization

Tokenization is a process used in Large Language Models to convert raw text into smaller and more semantically manageable units called *tokens*. In our case, the tokens will be as follows:
- `<cls>` : Start of sequence
- `<pad>` : Padding token
- `<unk>` : Unknown token
- `<eos>` : End of sequence
- Single-letter amino acid codes

The `AutoTokenizer` class from the `transformers` library allows us to automatically tokenize our amino acid sequences.

Below is an example of the tokenization protocol that is used in a generic protein language model.

<figure>
<img src="images/aa_tokenization.png" alt="A generic overview of amino acid tokenization" width="700">
<figcaption><strong>Figure 5:</strong> A generic overview of amino acid tokenization.</figcaption>
</figure>

### How to Generate Protein Embeddings

Based on what we've discussed, we need to do the following to generate a sequence:
1. Provide a valid amino acid sequence
2. Use a tokenizer to chunk the sequence
3. Run the tokenized sequence through a protein language model

We will do this together!

In [ ]:
!pip install -q transformers

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)
model = model.to(device)

In [ ]:
amino_acid_sequence = ""  # TODO: add your own sequence here

In [ ]:
def get_embedding(sequence):
    inputs = tokenizer(sequence, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        
        # last_hidden_state shape: [1, sequence_length, hidden_dim]
        last_hidden_state = outputs.last_hidden_state
        embedding = last_hidden_state.mean(dim=1) 
                
    # Return as a 1D numpy array
    return embedding.cpu().numpy().squeeze()

In [ ]:
aa_embedding = get_embedding(amino_acid_sequence)
print(aa_embedding)

If the model downloaded correctly, you'll find that we printed out a massive list of numbers. This list corresponds to features for the amino acid sequence you chose!

If it didn't work, don't worry! We will provide the embeddings for all the proteins that you will be working with in the graded questions.

## Introduction to Multi-Class Classification

In modules 6 and 7, we encountered **binary classification** — predicting one of two outcomes (e.g., "cancer" vs. "healthy"). In this module, our tasks involve **multi-class classification**, where we predict one of *three or more* categories.

For example, if we choose to predict the domain a protein comes from, our model must decide between three classes: **Eukaryote**, **Prokaryote**, or **Virus**.

### One-Hot Encoding

Before we can train a model, we need a numerical representation of our class labels. We do this using **one-hot encoding**, where each class is represented as a binary vector with a single `1` at the position corresponding to that class.

For our domain prediction task with 3 classes:

| Label       | One-Hot Vector |
|-------------|----------------|
| Eukaryote   | [1, 0, 0]     |
| Prokaryote  | [0, 1, 0]     |
| Virus       | [0, 0, 1]     |

This gives us a **target vector** $\mathbf{y}$ of length $C$ (the number of classes), where exactly one entry is 1 and the rest are 0.

### Logits to Probabilities with the Softmax Function for Neural Networks

Recall from modules 6 and 7 that we learned about the softmax function being used to convert the similarities between patches into probabilities in the attention mechanism. In multi-class classification, we will use softmax to transform the output of our model, **logits**, into a valid *probability distribution* over the classes.

<figure>
<img src="images/softmax.png" alt="Problem Statement" width="600">
<figcaption><strong>Figure 6:</strong> An example of making the outputs of a 7-class classification network into a probability distribution (made with Google Gemini).</figcaption>
</figure>

Softmax guarantees two useful properties:
1. Every output is between 0 and 1
2. All outputs sum to 1

This means we can interpret each output as the model's **confidence** that the input belongs to that class. In the case of classification, you would often take the class with the highest probability as the model's prediction.

### The Loss Function: Cross-Entropy

To train our model, we need a way to measure **how wrong** the model's predicted probabilities are compared to the true label. For multi-class classification, we use the **cross-entropy loss**:

$$\mathcal{L} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

where:
- $y_c$ is the true one-hot label for class $c$ (either 0 or 1)
- $\hat{y}_c$ is the predicted probability for class $c$ (from softmax)

Since only one entry in $\mathbf{y}$ is 1 (say, class $k$), this simplifies to:

$$\mathcal{L} = -\log(\hat{y}_k)$$

## Picking your Problem

You have the freedom to choose from 3 different problems for this final module! From a batch of protein sequences, you can choose to predict:
1. The subcellular localization of the protein
2. The function of the protein
3. The domain (eukaryote, prokaryote, virus) from which the protein comes

<figure>
<img src="images/problem_statement_2.png" alt="Problem Statement" width="1000">
<figcaption><strong>Figure 7:</strong> The overall flow for Module 8 (made with Google Gemini).</figcaption>
</figure>
